In [2]:
from yaml import safe_load
from datasets import load_dataset
config_path = "config/local_config.yaml"
config = safe_load(open(config_path))
data_params = config['data_params']
dataset = load_dataset("json", data_files=data_params["data_path"], split="train")
if data_params["select"] is not None:
    dataset = dataset.select(range(data_params["select"]))
config["solver_params"]["dataset"] = dataset

from model.solver import KaggleSolver
model = KaggleSolver(**config["solver_params"])


{'seed': 3407, 'peft_config': {'task_type': 'CAUSAL_LM', 'r': 8, 'lora_alpha': 32, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'lora_dropout': 0.05}, 'sft_config': {'dataset_text_field': 'text', 'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 4, 'warmup_steps': 5, 'num_train_epochs': 1, 'learning_rate': 0.0002, 'logging_steps': 1, 'optim': 'adamw_8bit', 'weight_decay': 0.001, 'lr_scheduler_type': 'linear', 'seed': 3407}, 'quantization_config': {'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True}, 'solver_params': {'model': '/kaggle/input/qwen3-8b-custom/transformers/default/1', 'data_path': '/kaggle/input/datasets/heymipolar/openr1-sample-3500/sample_3500.jsonl', 'base_url': None, 'api_key': None, 'max_seq_length': 65535, 'device_map': 'auto', 'train_before_inference': False, 'use_peft_train': True, 'peft_weight_save': 'lora_adapter', 'quantize_base_model': True, 'peft_config': {'ta

In [ ]:
import os

import kaggle_evaluation.aimo_3_inference_server
import pandas as pd
import polars as pl
from model.solver import KaggleSolver

model_path = "/kaggle/input/qwen3-8b-custom/transformers/default/1"
# model_path = "/mnt/workspace/model/qwen3-8b"
data_path = "/kaggle/input/datasets/heymipolar/openr1-sample-3500/sample_3500.jsonl"
max_seq_length = 2048
model = KaggleSolver(
    model_path,
    data_path,
    select=10,
    max_seq_length=max_seq_length,
    train_before_inference=True,
    trust_remote_code=True
)

# Replace this function with your inference code.
# The function should return a single integer between 0 and 99999, inclusive.
def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame | pd.DataFrame:
    """Make a prediction."""
    # Unpack values
    id_ = id_.item(0)
    problem_text: str = problem.item(0)
    # Make a prediction
    # The model is loaded on the first call
    prediction = model.predict(problem_text)
    return pl.DataFrame({'id': id_, 'answer': prediction})

inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(
    predict
)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # You MUST call this within 15 minutes of the script starting. This is to
    # ensure a "fast fail" in case a bug prevents the inference server from starting.
    # Do anything that might take a long time (like model loading) in the predict
    # function, which has no time limit.
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )
